# PYNQ-Z2 CNN 가속기 보드 검증

Vivado에서 만든 `pynq_z2_cnn_jupyter.zip`과 이 노트북을 Jupyter 홈에 업로드한 뒤, 셀을 위에서 아래로 실행합니다. 첫 테스트는 IC=1, 두 번째 테스트는 IC=3 누적을 검증합니다.

In [ ]:
from pathlib import Path
from zipfile import ZipFile

NOTEBOOK_DIR = Path.cwd()
ARCHIVE = NOTEBOOK_DIR / 'pynq_z2_cnn_jupyter.zip'
PROJECT_DIR = NOTEBOOK_DIR / 'pynq_z2_cnn'

if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Jupyter 홈에 업로드할 파일이 없습니다: {ARCHIVE.name}')

PROJECT_DIR.mkdir(exist_ok=True)
with ZipFile(ARCHIVE, 'r') as archive:
    archive.extractall(PROJECT_DIR)

print(f'압축 해제 완료: {PROJECT_DIR}')

In [ ]:
required = [
    PROJECT_DIR / 'pynq_z2_cnn.bit',
    PROJECT_DIR / 'pynq_z2_cnn.hwh',
    PROJECT_DIR / 'smoke_test_single_conv.py',
    PROJECT_DIR / 'single_conv_tile_28' / 'layer_config.json',
    PROJECT_DIR / 'multi_ic_conv_tile_28' / 'layer_config.json',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('누락된 파일:\n' + '\n'.join(missing))

for path in required:
    print('OK:', path.relative_to(PROJECT_DIR))

## 1) IC=1 기본 연결 테스트

PS DDR → DMA MM2S → PL 3×3 convolution → DMA S2MM → PS DDR 경로와 676개 INT32 결과를 확인합니다.

In [ ]:
import subprocess
import sys

single_command = [
    sys.executable,
    'smoke_test_single_conv.py',
    '--bitstream', 'pynq_z2_cnn.bit',
    '--fixture', 'single_conv_tile_28',
]
subprocess.run(single_command, cwd=PROJECT_DIR, check=True)

정상이라면 마지막 줄에 `PASS: PYNQ-Z2 AXI DMA smoke test input_channels=1 ... outputs=676`이 표시됩니다.

## 2) IC=3 partial-sum 누적 테스트

세 입력 채널을 순차 처리하고, 중간 채널은 DDR로 출력하지 않으며 마지막 채널에서만 676개 결과를 전송하는지 확인합니다.

In [ ]:
multi_command = [
    sys.executable,
    'smoke_test_single_conv.py',
    '--bitstream', 'pynq_z2_cnn.bit',
    '--fixture', 'multi_ic_conv_tile_28',
]
subprocess.run(multi_command, cwd=PROJECT_DIR, check=True)

정상이라면 마지막 줄에 `PASS: PYNQ-Z2 AXI DMA smoke test input_channels=3 ... outputs=676`이 표시됩니다. 실패하면 해당 셀의 `status`, `stream_in`, `stream_out`, `errors`와 예외 전체를 팀에 전달합니다.